In [25]:
import pandas as pd
import sqlite3

df = pd.read_csv("superstore.csv", encoding="latin1")
df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y")
df["ship_date"] = pd.to_datetime(df["ship_date"], format="%m/%d/%Y")

conn = sqlite3.connect("superstore.db")
df.to_sql("orders_raw", conn, if_exists="replace", index=False)
conn.close()

print(f"Loaded {len(df)} rows into orders_raw")

Loaded 9994 rows into orders_raw


In [26]:
conn = sqlite3.connect("superstore.db")
pd.read_sql("SELECT * FROM orders_raw LIMIT 5", conn)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [27]:
schema_sql = """
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;

CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT,
    segment       TEXT
);

CREATE TABLE products (
    product_id    TEXT PRIMARY KEY,
    product_name  TEXT,
    category      TEXT,
    sub_category  TEXT
);

CREATE TABLE orders (
    row_id        INTEGER PRIMARY KEY,
    order_id      TEXT,
    order_date    DATE,
    ship_date     DATE,
    ship_mode     TEXT,
    customer_id   TEXT REFERENCES customers(customer_id),
    product_id    TEXT REFERENCES products(product_id),
    country       TEXT,
    city          TEXT,
    state         TEXT,
    postal_code   TEXT,
    region        TEXT,
    sales         NUMERIC,
    quantity      INTEGER,
    discount      NUMERIC,
    profit        NUMERIC
);
"""

conn = sqlite3.connect("superstore.db")
conn.executescript(schema_sql)
conn.close()

In [28]:
populate_sql = """
INSERT INTO customers
SELECT customer_id, customer_name, segment
FROM orders_raw
GROUP BY customer_id;

INSERT INTO products
SELECT product_id, product_name, category, sub_category
FROM orders_raw
GROUP BY product_id;

INSERT INTO orders
SELECT row_id, order_id, order_date, ship_date, ship_mode,
       customer_id, product_id, country, city, state, postal_code, region,
       sales, quantity, discount, profit
FROM orders_raw;
"""

conn = sqlite3.connect("superstore.db")
conn.executescript(populate_sql)
conn.close()

In [29]:
conn = sqlite3.connect("superstore.db")
for t in ["customers", "products", "orders"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)["n"][0]
    print(t, n)
conn.close()

customers 793
products 1862
orders 9994


In [ ]:
q1 = """
WITH monthly_sales AS (
  SELECT strftime('%Y-%m', order_date) AS month,
         region,
         SUM(sales) AS total_sales
  FROM orders
  GROUP BY 1, 2
)
SELECT month, region, total_sales,
       LAG(total_sales) OVER (PARTITION BY region ORDER BY month) AS prev_month,
       ROUND(100.0 * (total_sales - LAG(total_sales) OVER (PARTITION BY region ORDER BY month))
             / NULLIF(LAG(total_sales) OVER (PARTITION BY region ORDER BY month), 0), 1) AS mom_growth_pct
FROM monthly_sales
ORDER BY region, month;
"""
conn = sqlite3.connect("superstore.db")
result1 = pd.read_sql(q1, conn)
display(result1.head(10))
conn.close()

,month,region,total_sales,prev_month,mom_growth_pct
0,2014-01,Central,1539.9060,NaN,NaN
1,2014-02,Central,1233.1740,1539.9060,-19.9
2,2014-03,Central,5827.6020,1233.1740,372.6
3,2014-04,Central,3712.3400,5827.6020,-36.3
4,2014-05,Central,4048.5060,3712.3400,9.1
5,2014-06,Central,9646.2986,4048.5060,138.3
6,2014-07,Central,6740.5740,9646.2986,-30.1
7,2014-08,Central,3022.1830,6740.5740,-55.2
8,2014-09,Central,34408.6898,3022.1830,1038.5
9,2014-10,Central,8965.7570,34408.6898,-73.9


In [66]:
q2 = """
WITH sub_cat_profit AS (
  SELECT p.category, p.sub_category,
         SUM(o.sales) AS total_sales,
         SUM(o.profit) AS total_profit
  FROM orders o
  JOIN products p ON o.product_id = p.product_id
  GROUP BY 1, 2
)
SELECT *, ROUND(100.0 * total_profit / NULLIF(total_sales,0), 1) AS profit_margin_pct
FROM sub_cat_profit
ORDER BY profit_margin_pct ASC;
"""
conn = sqlite3.connect("superstore.db")
result2 = pd.read_sql(q2, conn)
display(result2.head(10))
conn.close()

,category,sub_category,total_sales,total_profit,profit_margin_pct
0,Furniture,Tables,206965.5320,-17725.4811,-8.6
1,Furniture,Bookcases,114879.9963,-3472.5560,-3.0
2,Office Supplies,Supplies,46673.5380,-1189.0995,-2.5
3,Technology,Machines,189238.6310,3384.7569,1.8
4,Furniture,Chairs,328449.1030,26590.1663,8.1
5,Office Supplies,Storage,223843.6080,21278.8264,9.5
6,Technology,Phones,330007.0540,44515.7306,13.5
7,Furniture,Furnishings,91705.1640,13059.1436,14.2
8,Office Supplies,Binders,203412.7330,30221.7633,14.9
9,Office Supplies,Appliances,107532.1610,18138.0054,16.9


In [65]:
q3 = """
WITH customer_value AS (
  SELECT c.customer_id, c.customer_name, c.segment, SUM(o.sales) AS ltv
  FROM orders o
  JOIN customers c ON o.customer_id = c.customer_id
  GROUP BY 1, 2, 3
)
SELECT *, RANK() OVER (PARTITION BY segment ORDER BY ltv DESC) AS ltv_rank
FROM customer_value
ORDER BY segment, ltv_rank
LIMIT 30;
"""
conn = sqlite3.connect("superstore.db")
result3 = pd.read_sql(q3, conn)
display(result3.head(10))
conn.close()

,customer_id,customer_name,segment,ltv,ltv_rank
0,RB-19360,Raymond Buch,Consumer,15117.339,1
1,AB-10105,Adrian Barton,Consumer,14473.571,2
2,KL-16645,Ken Lonsdale,Consumer,14175.229,3
3,SC-20095,Sanjit Chand,Consumer,14142.334,4
4,HL-15040,Hunter Lopez,Consumer,12873.298,5
5,SE-20110,Sanjit Engle,Consumer,12209.438,6
6,CC-12370,Christopher Conant,Consumer,12129.072,7
7,GT-14710,Greg Tran,Consumer,11820.120,8
8,BM-11140,Becky Martin,Consumer,11789.630,9
9,SV-20365,Seth Vernon,Consumer,11470.950,10


In [52]:
q4 = """
SELECT p.sub_category, SUM(o.profit) AS total_profit
FROM orders o JOIN products p ON o.product_id = p.product_id
GROUP BY 1
ORDER BY total_profit ASC
LIMIT 5;
"""
conn = sqlite3.connect("superstore.db")
result4 = pd.read_sql(q4, conn)
display(result4)
conn.close()

,sub_category,total_profit
0,Tables,-17725.4811
1,Bookcases,-3472.5560
2,Supplies,-1189.0995
3,Fasteners,949.5182
4,Machines,3384.7569


In [53]:
q5 = """
SELECT
  CASE
    WHEN discount = 0 THEN '0%'
    WHEN discount <= 0.2 THEN '1-20%'
    WHEN discount <= 0.4 THEN '21-40%'
    ELSE '40%+'
  END AS discount_bucket,
  ROUND(AVG(profit / NULLIF(sales,0)) * 100, 1) AS avg_margin_pct,
  COUNT(*) AS n_orders
FROM orders
GROUP BY 1
ORDER BY 1;
"""
conn = sqlite3.connect("superstore.db")
result5 = pd.read_sql(q5, conn)
display(result5)
conn.close()

,discount_bucket,avg_margin_pct,n_orders
0,0%,34.0,4798
1,1-20%,17.4,3803
2,21-40%,-16.7,460
3,40%+,-108.9,933


In [64]:
q6 = """
SELECT c.customer_name, SUM(o.sales) AS ltv
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
GROUP BY 1
ORDER BY ltv DESC
LIMIT 10;
"""
conn = sqlite3.connect("superstore.db")
result6 = pd.read_sql(q6, conn)
display(result6)
conn.close()

,customer_name,ltv
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


In [63]:
q7 = """
WITH first_order AS (
  SELECT customer_id, MIN(order_date) AS first_date
  FROM orders GROUP BY 1
),
cohort AS (
  SELECT f.customer_id, strftime('%Y-%m', f.first_date) AS cohort_month,
         COUNT(DISTINCT o.order_id) AS n_orders
  FROM first_order f
  JOIN orders o ON o.customer_id = f.customer_id
  GROUP BY 1, 2
)
SELECT cohort_month,
       COUNT(*) AS customers,
       SUM(CASE WHEN n_orders > 1 THEN 1 ELSE 0 END) AS repeat_customers,
       ROUND(100.0 * SUM(CASE WHEN n_orders > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS repeat_rate_pct
FROM cohort
GROUP BY 1
ORDER BY 1;
"""
conn = sqlite3.connect("superstore.db")
result7 = pd.read_sql(q7, conn)
display(result7.head(10))
conn.close()

,cohort_month,customers,repeat_customers,repeat_rate_pct
0,2014-01,32,32,100.0
1,2014-02,24,24,100.0
2,2014-03,65,65,100.0
3,2014-04,56,56,100.0
4,2014-05,56,56,100.0
5,2014-06,48,48,100.0
6,2014-07,44,44,100.0
7,2014-08,49,49,100.0
8,2014-09,68,68,100.0
9,2014-10,42,42,100.0


In [55]:
q8 = """
SELECT ship_mode,
       ROUND(AVG(julianday(ship_date) - julianday(order_date)), 1) AS avg_delivery_days
FROM orders
GROUP BY 1
ORDER BY avg_delivery_days;
"""
conn = sqlite3.connect("superstore.db")
result8 = pd.read_sql(q8, conn)
display(result8)
conn.close()

,ship_mode,avg_delivery_days
0,Same Day,0.0
1,First Class,2.2
2,Second Class,3.2
3,Standard Class,5.0


In [56]:
q9 = """
SELECT p.category,
       SUM(o.sales) AS total_sales,
       SUM(o.profit) AS total_profit,
       ROUND(100.0 * SUM(o.profit) / NULLIF(SUM(o.sales),0), 1) AS margin_pct
FROM orders o JOIN products p ON o.product_id = p.product_id
GROUP BY 1
ORDER BY total_sales DESC;
"""
conn = sqlite3.connect("superstore.db")
result9 = pd.read_sql(q9, conn)
display(result9)
conn.close()

,category,total_sales,total_profit,margin_pct
0,Technology,836154.0330,145454.9481,17.4
1,Furniture,741999.7953,18451.2728,2.5
2,Office Supplies,719047.0320,122490.8008,17.0


In [62]:
q10 = """
WITH state_sales AS (
  SELECT state, SUM(sales) AS total_sales
  FROM orders GROUP BY 1
),
ranked AS (
  SELECT *, SUM(total_sales) OVER (ORDER BY total_sales DESC) AS running_total,
         SUM(total_sales) OVER () AS grand_total
  FROM state_sales
)
SELECT state, total_sales,
       ROUND(100.0 * running_total / grand_total, 1) AS cumulative_pct
FROM ranked
ORDER BY total_sales DESC
LIMIT 15;
"""
conn = sqlite3.connect("superstore.db")
result10 = pd.read_sql(q10, conn)
display(result10.head(10))
conn.close()

,state,total_sales,cumulative_pct
0,California,457687.6315,19.9
1,New York,310876.2710,33.5
2,Texas,170188.0458,40.9
3,Washington,138641.2700,46.9
4,Pennsylvania,116511.9140,52.0
5,Florida,89473.7080,55.9
6,Illinois,80166.1010,59.4
7,Ohio,78258.1360,62.8
8,Michigan,76269.6140,66.1
9,Virginia,70636.7200,69.2


In [58]:
q11 = """
SELECT c.segment,
       COUNT(DISTINCT o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS avg_orders_per_customer
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
GROUP BY 1;
"""
conn = sqlite3.connect("superstore.db")
result11 = pd.read_sql(q11, conn)
display(result11)
conn.close()

,segment,avg_orders_per_customer
0,Consumer,6.322738
1,Corporate,6.415254
2,Home Office,6.141892


In [61]:
q12 = """
SELECT strftime('%m', order_date) AS month_num,
       strftime('%Y', order_date) AS year,
       SUM(sales) AS total_sales
FROM orders
GROUP BY 1, 2
ORDER BY 1, 2;
"""
conn = sqlite3.connect("superstore.db")
result12 = pd.read_sql(q12, conn)
display(result12.head(10))
conn.close()

,month_num,year,total_sales
0,01,2014,14236.8950
1,01,2015,18174.0756
2,01,2016,18542.4910
3,01,2017,43971.3740
4,02,2014,4519.8920
5,02,2015,11951.4110
6,02,2016,22978.8150
7,02,2017,20301.1334
8,03,2014,55691.0090
9,03,2015,38726.2520


In [68]:
queries = {
    "Q1: Month-over-month sales growth by region": q1,
    "Q2: Profit margin by sub-category": q2,
    "Q3: Customer LTV ranked within segment": q3,
    "Q4: Top 5 loss-making sub-categories": q4,
    "Q5: Discount level vs avg profit margin": q5,
    "Q6: Top 10 customers by lifetime value": q6,
    "Q7: Repeat purchase rate by cohort month": q7,
    "Q8: Shipping mode vs avg delivery delay": q8,
    "Q9: Category profitability vs sales volume": q9,
    "Q10: State-level sales concentration (Pareto)": q10,
    "Q11: Order frequency per customer segment": q11,
    "Q12: Seasonal sales trend by month/year": q12,
}

with open("queries.sql", "w") as f:
    f.write("-- Superstore Retail Performance Analysis\n")
    f.write("-- Note: written for SQLite; swap strftime()/julianday() for\n")
    f.write("-- DATE_TRUNC()/date subtraction if porting to Postgres.\n\n")
    for title, sql in queries.items():
        f.write(f"-- {title}\n{sql.strip()}\n\n")

print("Saved queries.sql")

Saved queries.sql


In [69]:
with open("schema.sql", "w") as f:
    f.write("-- Superstore schema (Postgres-compatible DDL)\n\n")
    f.write(schema_sql.strip() + "\n")

print("Saved schema.sql")

Saved schema.sql


In [72]:
conn = sqlite3.connect("superstore.db")
for t in ["orders", "customers", "products"]:
    pd.read_sql(f"SELECT * FROM {t}", conn).to_csv(f"{t}.csv", index=False)
conn.close()